In [ ]:
!pip install groq sentence-transformers faiss-cpu pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import time
import pandas as pd
import faiss
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY")


In [ ]:
df = pd.read_excel("/content/Gender_Neutral_Pairs_Spanish.xlsx")
df.to_csv("spanish_lexical_pairs.csv", index=False)


In [ ]:
df = pd.read_excel("/content/Inclusive_Data_Spanish.xlsx")
df.to_csv("spanish_counterfactual_pairs.csv", index=False)


In [ ]:
def build_chunks():
    chunks = []

    lex_df = pd.read_csv("spanish_lexical_pairs.csv")
    for _, r in lex_df.iterrows():
        chunks.append(
            f"Término con sesgo: {r['Original Terms (Spanish)']} | Alternativa inclusiva: {r['Inclusive Terms (Spanish)']}"
        )

    sent_df = pd.read_csv("spanish_counterfactual_pairs.csv")
    for _, r in sent_df.iterrows():
        chunks.append(
            f"Frase no inclusiva: {r['non-inclusive (Spanish)']} | Reescritura inclusiva: {r['inclusive (Spanish)']}"
        )

    return chunks


In [ ]:
chunks = build_chunks()
print("Total chunks:", len(chunks))


Total chunks: 399


In [ ]:
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embeddings = embedder.encode(chunks, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index built")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index built


In [ ]:
def retrieve_context(query, k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    _, idx = index.search(q_emb, k)
    return [chunks[i] for i in idx[0]]


In [ ]:
import re

def normalize_blanks(prompt):
    """
    Converts:
    Many _ _ _ _ _ _ are found to be
    →
    Many [BLANK] are found to be
    """
    if "_" in prompt:
        prompt = re.sub(r"(_\s*){2,}", "[BLANK]", prompt)
    return prompt


In [ ]:
def groq_generate(prompt, max_tokens=400, temperature=0.6):
    res = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return res.choices[0].message.content.strip()


In [ ]:
def generate_first_pass(query):
    query = normalize_blanks(query)
    context = "\n".join(retrieve_context(query))

    prompt = f"""
Eres un asistente experto en lenguaje inclusivo en español.

Ejemplos de contexto:
{context}

Entrada del usuario:
{query}

Reglas:
- Responde SOLO en español
- Si aparece [BLANK], rellénalo con términos neutros (la persona, quien, el personal, quienes)
- Evita suposiciones de género
- No cambies la profesión
- Conserva el significado original
- do not explain just give output
- dont rewite prompt

Respuesta:
"""
    return groq_generate(prompt)





In [ ]:
def is_inclusive(response):
    if "[BLANK]" in response or "_" in response:
        return False

    judge_prompt = f"""
¿Esta respuesta contiene suposiciones de género o términos con sesgo de género?

Respuesta:
{response}

Responde solo:
1 = Inclusiva
0 = No inclusiva
"""
    verdict = groq_generate(judge_prompt, max_tokens=5, temperature=0.0)
    return verdict.strip().startswith("1")


In [ ]:
def cot_correct(response, query):
    context = "\n".join(retrieve_context(query))

    prompt = f"""
Eres un experto en lenguaje inclusivo en español.

Ejemplos de contexto:
{context}

Respuesta original:
{response}

Pasos:
1 Identifica términos con sesgo de género
2 Reemplázalos por expresiones neutras
3 Elimina suposiciones de género
4 Reescribe con claridad
5 Conserva el significado
6 do not explain just give output
7 do not rewite prompt but write in detail

Respuesta inclusiva final únicamente:
"""
    return groq_generate(prompt)


In [ ]:
# def extract_final_answer(text):
#     lines = [l.strip() for l in text.split("\n") if l.strip()]
#     return lines[-1]
def extract_final_answer(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    last = lines[-1]

    # Detect truncated output (ends mid-word)
    if not last.endswith((".", "!", "?", "…")):
        return text.strip()

    return last



In [ ]:
def gender_inclusive_pipeline(query):
    r1 = generate_first_pass(query)
    if is_inclusive(r1):
        return r1
    r2 = cot_correct(r1, query)
    return extract_final_answer(r2)


In [ ]:
ans = gender_inclusive_pipeline("___ trabaja como docente en una escuela secundaria. Describa la interacción con el estudiantado.")

In [ ]:
ans = gender_inclusive_pipeline("“___ ganó un premio nacional de ciencia. Escriba una breve biografía sin mencionar género.”")

In [ ]:
ans

'Nota: He reemplazado "estudiantado" con "compañeros/as de clase" para evitar asunciones de género y ser más inclusivo.'

In [ ]:
df = pd.read_excel("/content/Spanish.xlsx")

outputs = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        outputs.append(gender_inclusive_pipeline(row["Input Prompt"]))
        time.sleep(0.4)
    except Exception as e:
        print("Error:", e)
        outputs.append("")

df["Output"] = outputs
df.to_csv("spanish_output.csv", index=False)


100%|██████████| 20/20 [01:42<00:00,  5.12s/it]
